# 00 — Full analysis, start to finish

This notebook reproduces **every step actually run** to produce the results in the README, in the order they were run. Notebooks 01–07 break the same work into themed chapters; this one is the complete record in a single file.

Runtime is roughly 4–6 minutes, dominated by the model sweep and permutation importance.

**Expected outputs are stated in the markdown before each step**, so you can tell immediately if a result has drifted.

## 0. Setup

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
from smartnet import config
from smartnet.data import loader, validation
from smartnet.evaluation.splits import summarise_strategies
from smartnet.evaluation import metrics as M
from smartnet.models.experiment import run_cv, run_sweep, leakage_table
from smartnet.models.interpret import permutation_importances, block_importance
from smartnet.visualization import plots as P

print('Label schemes :', list(config.LABEL_MAPS))
print('Features      :', len(config.ALL_FEATURES))
print('Seed          :', config.RANDOM_SEED)

## 1. Load and derive time structure

Expect **1,340 rows x 53 columns**, resolving to **664 events across 25 recording days**.

In [ ]:
df = loader.load_analysis_frame()
print(df.shape)
print('Period :', df.timestamp.min(), '->', df.timestamp.max())
print('Events :', df.event_id.nunique(), '| Days :', df.session_date.nunique())

### Why events matter

Labelled motions are contiguous runs of 1-second epochs. This is the grouping unit for cross-validation, and it is only defensible if every run carries a single label — so that is asserted, not assumed.

In [ ]:
gaps = df.sort_values('timestamp')['gap_seconds'].dropna()
print('Gaps of exactly 1s :', int((gaps == 1).sum()), 'of', len(gaps))
loader.assert_events_are_label_pure(df)
print('All events label-pure.')

events = loader.event_summary(df, 'motion_5cat')
print('\nEvent length (epochs):')
print(events.n_epochs.describe()[['count','50%','max']].to_string())

## 2. Data quality

Eight checks. Expect **all pass**, with `class_support` reporting the rarest class at 91 epochs for the 4-category scheme.

In [ ]:
report = validation.run_all(df)
print(report.to_string(index=False))
print('\nBlocking failure:', validation.has_blocking_failure(report))

## 3. Class balance

The imbalance is the central constraint: **Enter (47) and Exit (44)** are the rarest classes and the most operationally interesting.

In [ ]:
for col in ['motion_4cat', 'motion_5cat']:
    names = config.LABEL_MAPS[col]
    vc = df[col].value_counts().sort_index()
    print(f'--- {col}')
    for k, v in vc.items():
        print(f'    {names[int(k)]:<16} {v:5d}  ({100*v/len(df):5.1f}%)')

## 4. Leakage diagnostic

Each row carries features for the 10 seconds either side, so adjacent rows within an event are near-duplicates.

Expect: **59.6%** of test epochs under a random split come from an event that is also in training. Grouped splits: **0.0%**.

In [ ]:
diag = summarise_strategies(df, 'motion_5cat')
print(diag[['strategy','grouping','shared_events','pct_test_from_seen_event']].to_string(index=False))

In [ ]:
fig = P.plot_leakage_diagnostic(diag); plt.show()

## 5. Model sweep

Five models x three validation designs. Slowest cell in the notebook (~90s).

Expect inflation of **0.003 to 0.010** — i.e. random splitting inflates accuracy by well under a percentage point.

In [ ]:
results5, artefacts5 = run_sweep(df, label_col='motion_5cat')
leakage_table(results5)

### The hypothesis was wrong

I built this expecting the random split to inflate results substantially. It does not. The structural risk is real and measurable — 59.6% shared events — but the classes are separated by broad signal-energy differences that generalise across events and days, not by event-specific quirks a model could memorise.

This is reported as a negative result rather than quietly dropped. The design is still correct: the diagnostic is cheap, the risk was genuine, and on a dataset with longer events or subtler classes the answer could have gone the other way.

**Grouped-event CV is used from here on.**

In [ ]:
print(results5[results5.strategy=='grouped_event'][
    ['model','cv_accuracy_mean','cv_balanced_accuracy_mean','cv_macro_f1_mean','cv_macro_auc_mean']
].round(4).to_string(index=False))

## 6. The headline gap

Expect **94.7% accuracy** against **78.7% balanced accuracy** for the random forest — a 16-point gap.

In [ ]:
fig = P.plot_accuracy_vs_balanced(results5); plt.show()

In [ ]:
names = config.LABEL_MAPS['motion_5cat']
ordered = [names[k] for k in sorted(names)]
art = artefacts5['motion_5cat|random_forest|grouped_event']
valid = ~np.isnan(art['oof_pred'])
y_true = np.array([names[int(v)] for v in art['y'][valid]])
y_pred = np.array([names[int(v)] for v in art['oof_pred'][valid]])

per_class = M.per_class_sensitivity_specificity(y_true, y_pred, ordered)
print(per_class.round(3).to_string(index=False))

**Enter 0.34, Exit 0.61.** The two behaviours that matter most for malaria exposure are the ones the model handles worst, and overall accuracy conceals it entirely.

In [ ]:
cm5 = M.confusion_frame(y_true, y_pred, ordered)
fig = P.plot_confusion(cm5, 'Random forest, grouped-event CV — 5 categories'); plt.show()

## 7. Error analysis

Expect entry/exit confusion to account for the large majority of errors on those two classes.

In [ ]:
m = cm5.to_numpy()
i_en, i_ex = ordered.index('Enter'), ordered.index('Exit')
en_err = m[i_en].sum() - m[i_en, i_en]
ex_err = m[i_ex].sum() - m[i_ex, i_ex]
print(f'Enter: {en_err} errors, {m[i_en,i_ex]} predicted as Exit  ({100*m[i_en,i_ex]/en_err:.0f}%)')
print(f'Exit : {ex_err} errors, {m[i_ex,i_en]} predicted as Enter ({100*m[i_ex,i_en]/ex_err:.0f}%)')

Entering and exiting are confused **with each other**, not with the other behaviours.

This independently reproduces the limitation in Koudou et al. (2022), who reported entry/exit confusion accounting for 83.3% and 85.7% of errors on those classes — on different data, three years earlier, with a different feature set.

That convergence points to a physical limit rather than a modelling problem: one accelerometer on a net's side panel registers a similar disturbance whether a body moves in or out. **Direction is largely absent from the signal.**

## 8. Collapsing entry and exit

Expect the 4-category model to reach **0.978 accuracy / 0.965 balanced accuracy**, with the combined class at **0.934 sensitivity**.

In [ ]:
results4, artefacts4 = run_sweep(df, label_col='motion_4cat')
rf4 = results4[(results4.strategy=='grouped_event') & (results4.model=='random_forest')].iloc[0]
rf5 = results5[(results5.strategy=='grouped_event') & (results5.model=='random_forest')].iloc[0]

pd.DataFrame({
    '5-category': [rf5.cv_accuracy_mean, rf5.cv_balanced_accuracy_mean, rf5.cv_macro_f1_mean],
    '4-category': [rf4.cv_accuracy_mean, rf4.cv_balanced_accuracy_mean, rf4.cv_macro_f1_mean],
}, index=['accuracy','balanced_accuracy','macro_f1']).round(4)

In [ ]:
names4 = config.LABEL_MAPS['motion_4cat']
ordered4 = [names4[k] for k in sorted(names4)]
a4 = artefacts4['motion_4cat|random_forest|grouped_event']
v4 = ~np.isnan(a4['oof_pred'])
pc4 = M.per_class_sensitivity_specificity(
    np.array([names4[int(v)] for v in a4['y'][v4]]),
    np.array([names4[int(v)] for v in a4['oof_pred'][v4]]), ordered4)
print(pc4.round(3).to_string(index=False))
fig = P.plot_per_class(pc4, 'Per-class — 4 categories (grouped-event CV)'); plt.show()

Merging entry and exit lifts sensitivity on that behaviour from 0.34/0.61 to **0.934**.

**The four-category model is the recommended operating point.** It supports counting net crossings and detecting when a net goes up or down — enough for net-use duration and timing. It does not support directional inference and must not be used for it.

## 9. Feature ablation

Expect **8 features (current + aggregates) to beat all 48** on balanced accuracy, and backward-only to collapse to ~0.59.

In [ ]:
rows = []
for block in config.FEATURE_BLOCKS:
    r = run_cv(df, 'motion_5cat', 'random_forest', 'grouped_event',
               features=config.FEATURE_BLOCKS[block])
    rows.append({'block': block, 'n_features': r['n_features'],
                 'accuracy': round(r['cv_accuracy_mean'], 4),
                 'balanced_accuracy': round(r['cv_balanced_accuracy_mean'], 4)})
pd.DataFrame(rows).sort_values('balanced_accuracy', ascending=False)

Two consequences.

**Eight features outperform forty-eight.** The individual per-second lag columns add nothing — the feature set can be cut by 83%, which matters for on-device inference in field conditions.

**Forward context is essential.** Dropping it collapses balanced accuracy from 0.797 to 0.593. Knowing what happens *after* the tagged second is what separates a net going up from a net going down — so this classifier is inherently retrospective and needs ~10 seconds of lookahead. It cannot run in true real time. That constraint would otherwise only have surfaced in production.

## 10. Interpretation

Slow cell (~15s). Expect the **10-epoch aggregates to dominate**, led by `meanzover10vector_back`.

In [ ]:
imp = permutation_importances(df, 'motion_5cat', n_repeats=5, max_folds=2)
print(imp.head(10).round(4).to_string(index=False))
print()
print(block_importance(imp).to_string(index=False))

In [ ]:
fig = P.plot_feature_importance(imp); plt.show()

The window aggregates carry the signal; the 40 individual lag features contribute nothing (permuting them slightly *improves* held-out balanced accuracy, consistent with noise).

This matches the published importance analysis, which found averages over a single dimension across the two 10-second periods to be the top variables. Both point the same way: the discriminating information is the **aggregate orientation change of the net over a window**, not the fine structure within it. Raising or lowering a net produces a sustained displacement; a body crossing produces a burst. Those separate. Direction does not.

**Prediction is not causation.** These are associations between engineered signal features and a human-applied label, nothing more.

## 11. Summary of what this run established

| Finding | Value |
|---|---|
| Test epochs sharing an event with training, random split | 59.6% |
| Leakage inflation of accuracy | 0.3–1.0 pp *(hypothesis not supported)* |
| 5-category accuracy / balanced accuracy | 0.947 / 0.787 |
| Enter / Exit sensitivity | 0.34 / 0.61 |
| 4-category accuracy / balanced accuracy | 0.978 / 0.965 |
| Combined enter-or-exit sensitivity | 0.934 |
| Best feature block | 8 aggregates, beating all 48 |
| Balanced accuracy without forward context | 0.593 |

### Limitations this run cannot address

1. **No participant identifier**, so performance on a *new person* cannot be estimated — the question that matters most for deployment.
2. **No demographics**, so the adult/child fairness comparison reported in the published study cannot be reproduced.
3. **44–47 epochs** in the minority classes means roughly 9 test examples per fold; those per-class figures carry wide uncertainty.
4. **Staged daytime conditions**, not natural overnight net use.

The first two are blocked by the extract, not by method. A participant-linked extract from the PI would be the single biggest upgrade to this project.